In [ ]:
from pathlib import Path
import json
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/ubuntu/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
def make_coco_yaml_for_yolo(dataset_root: str, out_name: str = "dataset.yaml") -> Path:
    """
    Creates a Ultralytics-compatible dataset YAML file for a COCO-format dataset.
        - Expects the layout to be as described in the README
        - Assumes COCO category ids are 0..nc-1.
    """
    root = Path(dataset_root).expanduser().resolve()

    train_json = root / "annotations" / "instances_train.json"
    val_json   = root / "annotations" / "instances_val.json"

    # Read categories from train JSON (fallback to val if needed)
    coco = json.loads(train_json.read_text(encoding="utf-8")) if train_json.exists() \
        else json.loads(val_json.read_text(encoding="utf-8"))

    cats = coco.get("categories", [])
    if not cats:
        raise ValueError("No 'categories' found in COCO JSON.")

    # Build names indexed by category id
    id_to_name = {int(c["id"]): str(c["name"]) for c in cats}
    max_id = max(id_to_name.keys())
    missing = [i for i in range(max_id + 1) if i not in id_to_name]
    if missing:
        raise ValueError(f"Category ids are not contiguous from 0. Missing: {missing}")

    names = [id_to_name[i] for i in range(max_id + 1)]
    nc = len(names)

    def yaml_escape(s: str) -> str:
        return s.replace('"', '\\"')

    yaml_path = root / out_name
    yaml_text = "\n".join([
        f"path: {root.as_posix()}",
        "",
        "train: images/train",
        "val: images/val",
        "",
        "train_annotations: annotations/instances_train.json",
        "val_annotations: annotations/instances_val.json",
        "",
        f"nc: {nc}",
        "names:",
        *[f'  {i}: "{yaml_escape(n)}"' for i, n in enumerate(names)],
        "",
    ])


    yaml_path.write_text(yaml_text, encoding="utf-8")
    print(f"Wrote {yaml_path}")

In [10]:
make_coco_yaml_for_yolo("../Data/tomatoes")

Wrote /home/ubuntu/Thesis_WDV/Data/tomatoes/dataset.yaml


In [ ]:
# Load a model
model = YOLO("yolo11n.pt")

# Train the model
train_results = model.train(
    data="../Data/tomatoes/dataset.yaml",  # path to dataset YAML
    epochs=100,  # number of training epochs
    imgsz=640,  # training image size
    device="cpu",  # device to run on, i.e. device=0 or device=0,1,2,3 or device=cpu
)

# Evaluate model performance on the validation set
metrics = model.val()

# Perform object detection on an image
results = model("../Data/tomatoes/images/val/2023_08_02__13_32_31_894227000___02Z8R__02Z8W_bunch_1.png")
results[0].show()

# Export the model to ONNX format
path = model.export(format="onnx")  # return path to exported model

Ultralytics 8.3.237 🚀 Python-3.11.14 torch-2.9.1 CPU (Intel Xeon Platinum 8259CL CPU @ 2.50GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../Data/tomatoes/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots